# Limpieza de datos y estadísticas descriptivas — AlertaSegura Perú

Sprint 2 · Data Analysis

1. Corre el pipeline de limpieza (`scripts/limpieza_datos.py`) sobre
   `data/reportes_sucios.csv` — una versión con nulos, duplicados y
   formatos inconsistentes inyectados a propósito para probar el script.
2. Calcula las primeras estadísticas descriptivas sobre el dataset ya
   limpio.

## 1. Limpieza de datos

In [1]:
import sys
sys.path.append("../scripts")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from limpieza_datos import limpiar

sns.set_theme(style="whitegrid")

sucio = pd.read_csv("../data/reportes_sucios.csv")
print("Filas:", len(sucio))
sucio.isnull().sum()

Filas: 128


id             0
usuario_id     0
categoria      2
descripcion    3
latitud        0
longitud       0
distrito       6
estado         5
created_at     0
dtype: int64

In [2]:
print("Duplicados exactos (excluyendo id):", sucio.duplicated(subset=[c for c in sucio.columns if c != "id"]).sum())
sucio["distrito"].unique()[:10]

Duplicados exactos (excluyendo id): 5


<StringArray>
[                 'CALLAO',           'Puente Piedra',
              'Miraflores',                       nan,
                     'Ate', 'VILLA MARÍA DEL TRIUNFO',
                  'Callao',  'SAN JUAN DE LURIGANCHO',
          '  San Isidro  ',  'San Juan de Miraflores']
Length: 10, dtype: str

In [3]:
reportes, reporte_limpieza = limpiar(sucio)

print(f"Filas antes: {reporte_limpieza['filas_iniciales']}  →  Filas después: {reporte_limpieza['filas_finales']}")
reporte_limpieza

Filas antes: 128  →  Filas después: 114


{'filas_iniciales': 128,
 'fechas_invalidas': 0,
 'coordenadas_fuera_de_rango': 3,
 'duplicados_eliminados': 6,
 'nulos_por_columna_antes': {'id': 0,
  'usuario_id': 0,
  'categoria': 2,
  'descripcion': 3,
  'latitud': 3,
  'longitud': 3,
  'distrito': 6,
  'estado': 5,
  'created_at': 0},
 'filas_descartadas_por_nulos_obligatorios': 8,
 'filas_finales': 114}

In [4]:
# Verificación: ya no quedan nulos ni duplicados, y el texto está normalizado
assert reportes.isnull().sum().sum() == 0
assert reportes.duplicated(subset=[c for c in reportes.columns if c != "id"]).sum() == 0
sorted(reportes["distrito"].unique())

['Ate',
 'Callao',
 'Cercado de Lima',
 'Chorrillos',
 'Comas',
 'Desconocido',
 'Independencia',
 'La Victoria',
 'Los Olivos',
 'Miraflores',
 'Puente Piedra',
 'Rímac',
 'San Isidro',
 'San Juan de Lurigancho',
 'San Juan de Miraflores',
 'San Martín de Porres',
 'Villa El Salvador',
 'Villa María del Triunfo']

In [5]:
reportes["created_at"] = pd.to_datetime(reportes["created_at"])
reportes.to_csv("../data/reportes_limpios.csv", index=False)
reportes.head()

,id,usuario_id,categoria,descripcion,latitud,longitud,distrito,estado,created_at
0,56,37,Alumbrado Deficiente,Farol malogrado en el parque,-12.065982,-77.121065,Callao,PENDIENTE,2026-06-19 21:53:00
1,41,12,Acoso,Acoso a estudiantes cerca del colegio,-11.876435,-77.074469,Puente Piedra,PENDIENTE,2026-06-14 20:20:00
2,20,7,Robo,Sustrajeron mi celular en la calle,-12.116536,-77.023988,Miraflores,VERIFICADO,2026-05-14 03:12:00
3,32,19,Acoso,Un sujeto acosó a una mujer en la vereda,-11.964548,-77.076728,Desconocido,PENDIENTE,2026-07-21 21:32:00
4,99,35,Robo,Sustrajeron mi celular en la calle,-12.038605,-76.913012,Ate,PENDIENTE,2026-06-06 23:47:00


## 2. Estadísticas descriptivas

In [6]:
reportes.describe(include="all")

,id,usuario_id,categoria,descripcion,latitud,longitud,distrito,estado,created_at
count,114.000000,114.000000,114,114,114.000000,114.000000,114,114,114
unique,NaN,NaN,5,15,NaN,NaN,18,3,NaN
top,NaN,NaN,Acoso,Amenaza a comerciante local,NaN,NaN,Callao,PENDIENTE,NaN
freq,NaN,NaN,27,12,NaN,NaN,12,56,NaN
mean,59.719298,20.596491,NaN,NaN,-12.046049,-77.030709,NaN,NaN,2026-06-16 02:27:46.315789
min,1.000000,1.000000,NaN,NaN,-12.226211,-77.127755,NaN,NaN,2026-04-06 08:30:00
25%,29.250000,9.250000,NaN,NaN,-12.121836,-77.075667,NaN,NaN,2026-05-24 23:53:30
50%,59.500000,20.500000,NaN,NaN,-12.047125,-77.034596,NaN,NaN,2026-06-12 20:24:30
75%,89.750000,30.000000,NaN,NaN,-11.977105,-77.010797,NaN,NaN,2026-07-01 17:49:30
max,120.000000,40.000000,NaN,NaN,-11.860502,-76.913012,NaN,NaN,2026-10-06 07:04:00


In [7]:
print("Total de reportes:", len(reportes))
print("Distritos distintos:", reportes["distrito"].nunique())
print("Categorías distintas:", reportes["categoria"].nunique())
print("Rango de fechas:", reportes["created_at"].min(), "→", reportes["created_at"].max())

Total de reportes: 114
Distritos distintos: 18
Categorías distintas: 5
Rango de fechas: 2026-04-06 08:30:00 → 2026-10-06 07:04:00


In [8]:
# Reportes promedio por distrito, y distrito con más/menos reportes
conteo_distrito = reportes["distrito"].value_counts()
print("Promedio de reportes por distrito:", round(conteo_distrito.mean(), 1))
print("Distrito con más reportes:", conteo_distrito.idxmax(), f"({conteo_distrito.max()})")
print("Distrito con menos reportes:", conteo_distrito.idxmin(), f"({conteo_distrito.min()})")

Promedio de reportes por distrito: 6.3
Distrito con más reportes: Callao (12)
Distrito con menos reportes: San Isidro (1)


In [9]:
# Distribución de categorías (%)
(reportes["categoria"].value_counts(normalize=True) * 100).round(1)

categoria
Acoso                   23.7
Bache                   21.9
Extorsión               21.9
Alumbrado Deficiente    20.2
Robo                    12.3
Name: proportion, dtype: Float64

In [10]:
# Tasa de verificación global
(reportes["estado"].value_counts(normalize=True) * 100).round(1)

estado
PENDIENTE     49.1
VERIFICADO    37.7
RECHAZADO     13.2
Name: proportion, dtype: Float64

In [11]:
# Tabla cruzada: categoría x estado (conteo)
pd.crosstab(reportes["categoria"], reportes["estado"])

estado,PENDIENTE,RECHAZADO,VERIFICADO
categoria,,,
Acoso,14,1,12
Alumbrado Deficiente,13,4,6
Bache,12,3,10
Extorsión,9,6,10
Robo,8,1,5


In [12]:
reportes["hora"] = reportes["created_at"].dt.hour
reportes.groupby("categoria")["hora"].agg(["mean", "median", "std"]).round(1)

,mean,median,std
categoria,,,
Acoso,13.1,19.0,9.0
Alumbrado Deficiente,10.3,9.0,8.1
Bache,9.6,7.0,8.7
Extorsión,9.2,8.0,7.6
Robo,13.7,15.5,8.4


## Próximos pasos (Sprint 3)

- Generar un dataset sintético más grande y realista (150-300 registros).
- Primeros gráficos formales: reportes por distrito y por tipo de incidente
  (ya cubiertos parcialmente en `01_exploracion_inicial.ipynb`, se refinan aquí).